### dbt_exploration_3

In [1]:
import os
import pandas as pd
import snowflake.connector
from dotenv import load_dotenv
from IPython.display import display

pd.set_option('display.max_colwidth', 80)
pd.set_option('display.max_rows', 100)

load_dotenv()

conn = snowflake.connector.connect(
    account=os.getenv("SNOWFLAKE_ACCOUNT"),
    user=os.getenv("SNOWFLAKE_USER"),
    password=os.getenv("SNOWFLAKE_PASSWORD"),
    role=os.getenv("SNOWFLAKE_ROLE"),
    warehouse=os.getenv("SNOWFLAKE_WAREHOUSE"),
    database=os.getenv("SNOWFLAKE_DATABASE"),
)

def run_query(sql: str) -> pd.DataFrame:
    cur = conn.cursor()
    cur.execute(sql)
    rows = cur.fetchall()
    cols = [desc[0] for desc in cur.description]
    cur.close()
    return pd.DataFrame(rows, columns=cols)

print("Connected!")

Connected!


## 1. Raw Table Profiling & Investigating

For each source table, we:
1. Discover every key present in `RAW_PAYLOAD` dynamically using `OBJECT_KEYS()`
2. Count how often each key is non-null across all rows
3. Pull one representative sample value per key

This tells us the true shape of each source's payload — including fields the docs may not mention.

In [2]:
def profile_raw_source(database: str, schema: str, table: str) -> pd.DataFrame:
    full_table = f"{database}.{schema}.{table}"

    # Pull all payloads into Python as JSON strings
    raw_df = run_query(f"SELECT RAW_PAYLOAD::STRING AS payload FROM {full_table}")
    total_rows = len(raw_df)

    # Parse JSON and discover all keys in Python
    import json
    all_keys = set()
    parsed = []
    for row in raw_df['PAYLOAD']:
        try:
            obj = json.loads(row)
            parsed.append(obj)
            all_keys.update(obj.keys())
        except Exception:
            parsed.append({})

    # For each key: count non-nulls and grab a sample value
    records = []
    for key in sorted(all_keys):
        values = [p.get(key) for p in parsed]
        non_null = [v for v in values if v is not None]
        sample = str(non_null[0])[:120] if non_null else None
        records.append({
            'PAYLOAD_KEY': key,
            'NON_NULL_COUNT': len(non_null),
            'total_rows': total_rows,
            'population_pct': round(len(non_null) / total_rows * 100, 1),
            'SAMPLE_VALUE': sample,
        })

    return (
        pd.DataFrame(records)
        .sort_values('population_pct', ascending=False)
        .reset_index(drop=True)
    )

### JSearch Profiling

In [3]:
print('Profiling JSearch raw payload...')
jsearch_profile = profile_raw_source('RAW', 'JSEARCH', 'SRC_POSTINGS')

print(f"Total rows: {jsearch_profile['total_rows'].iloc[0]}")
print(f"Unique payload keys discovered: {len(jsearch_profile)}")
print()
display(jsearch_profile.drop(columns=['total_rows']))

Profiling JSearch raw payload...
Total rows: 136
Unique payload keys discovered: 34



,PAYLOAD_KEY,NON_NULL_COUNT,population_pct,SAMPLE_VALUE
0,apply_options,136,100.0,[{'apply_link': 'https://www.linkedin.com/jobs/view/data-analyst-new-york-at...
1,job_description,136,100.0,Jobright is a next-generation AI job search platform built to make career na...
2,job_publisher,136,100.0,LinkedIn
3,job_posted_at_timestamp,136,100.0,1779260400
4,job_posted_at_datetime_utc,136,100.0,2026-05-20T07:00:00.000Z
5,job_posted_at,136,100.0,18 hours ago
6,job_location,136,100.0,"New York, NY"
7,job_id,136,100.0,n7ha1uBvkMfQoGz5AAAAAA==
8,job_highlights,136,100.0,{}
9,job_google_link,136,100.0,https://www.google.com/search?q=jobs&gl=us&hl=en&udm=8#vhid=vt%3D20/docid%3D...


### JSearch Investigation

In [6]:
# JSearch: what's actually inside job_highlights when it exists?
#    We know it's {} but is it ever populated at all?
df = run_query("""
    SELECT RAW_PAYLOAD:job_highlights::STRING AS highlights
    FROM RAW.JSEARCH.SRC_POSTINGS
    WHERE RAW_PAYLOAD:job_highlights::STRING != '{}'
    LIMIT 5
""")
display(df)

,HIGHLIGHTS


In [12]:
# JSearch: the job_salary=65 anomaly — is that hourly bleed-through?
df = run_query("""
    SELECT 
        RAW_PAYLOAD:job_salary::STRING AS job_salary,
        RAW_PAYLOAD:job_salary_period::STRING AS period,
        RAW_PAYLOAD:job_min_salary::STRING AS min_salary,
        RAW_PAYLOAD:job_title::STRING AS title
    FROM RAW.JSEARCH.SRC_POSTINGS
    WHERE RAW_PAYLOAD:job_salary IS NOT NULL
""")
display(df)

,JOB_SALARY,PERIOD,MIN_SALARY,TITLE
0,NaN,NaN,NaN,Data Analyst (New York)
1,NaN,NaN,NaN,Data Analyst 1 - 32409
2,NaN,YEAR,74160,Data Science Analyst - UG Intern Conversion
3,NaN,NaN,NaN,Data Analyst- Global Merchandising Services (remote)
4,NaN,NaN,NaN,Entry Level Data Scientist/Analyst/Java full stack developer-remote
...,...,...,...,...
131,NaN,YEAR,145200,Lead Clinical Data Warehouse Engineer
132,NaN,NaN,NaN,Snowflake Data Engineer - W2 only
133,NaN,NaN,NaN,Azure Databricks Platform Engineer
134,NaN,HOUR,70,Data Engineer


In [18]:
# JSearch salary — full picture of what's actually populated and consistent
import json

df = run_query("SELECT RAW_PAYLOAD::STRING AS payload FROM RAW.JSEARCH.SRC_POSTINGS")
parsed = [json.loads(r) for r in df['PAYLOAD']]

salary_records = []
for p in parsed:
    salary_records.append({
        'job_title':        p.get('job_title'),
        'job_salary':       p.get('job_salary'),         # the mystery field
        'job_min_salary':   p.get('job_min_salary'),
        'job_max_salary':   p.get('job_max_salary'),
        'job_salary_period': p.get('job_salary_period'),
        'job_salary_string': p.get('job_salary_string'),
    })

salary_df = pd.DataFrame(salary_records)

print(f"Total rows: {len(salary_df)}")
print(f"\njob_salary non-null: {salary_df['job_salary'].notna().sum()}")
print(f"job_min_salary non-null: {salary_df['job_min_salary'].notna().sum()}")
print(f"job_max_salary non-null: {salary_df['job_max_salary'].notna().sum()}")
print(f"job_salary_period non-null: {salary_df['job_salary_period'].notna().sum()}")
print(f"job_salary_string non-null: {salary_df['job_salary_string'].notna().sum()}")

print("\njob_salary_period value counts (including nulls):")
print(salary_df['job_salary_period'].value_counts(dropna=False))

print("\nRows where job_salary IS populated:")
display(salary_df[salary_df['job_salary'].notna()])

print("\nRows where salary_period = HOUR:")
display(salary_df[salary_df['job_salary_period'] == 'HOUR'])

print("\nRows where min_salary populated but no period:")
display(salary_df[
    salary_df['job_min_salary'].notna() & salary_df['job_salary_period'].isna()
][['job_title', 'job_min_salary', 'job_max_salary', 'job_salary_period', 'job_salary_string']])

Total rows: 136

job_salary non-null: 2
job_min_salary non-null: 39
job_max_salary non-null: 39
job_salary_period non-null: 41
job_salary_string non-null: 41

job_salary_period value counts (including nulls):
job_salary_period
NaN     95
YEAR    37
HOUR     4
Name: count, dtype: int64

Rows where job_salary IS populated:


,job_title,job_salary,job_min_salary,job_max_salary,job_salary_period,job_salary_string
8,Data Management - Data Analyst-IT III,65.0,NaN,NaN,HOUR,65 an hour
52,"Data Analyst ,Purchase, NY; Florham Park, NJ; Remote will be considered- Imm...",70.0,NaN,NaN,HOUR,70 an hour



Rows where salary_period = HOUR:


,job_title,job_salary,job_min_salary,job_max_salary,job_salary_period,job_salary_string
8,Data Management - Data Analyst-IT III,65.0,NaN,NaN,HOUR,65 an hour
44,Data Analyst I,NaN,30.0,36.0,HOUR,30–36 an hour
52,"Data Analyst ,Purchase, NY; Florham Park, NJ; Remote will be considered- Imm...",70.0,NaN,NaN,HOUR,70 an hour
134,Data Engineer,NaN,70.0,100.0,HOUR,70–100 an hour



Rows where min_salary populated but no period:


,job_title,job_min_salary,job_max_salary,job_salary_period,job_salary_string


### TheirStack Profiling

In [9]:
print('Profiling TheirStack raw payload...')
theirstack_profile = profile_raw_source('RAW', 'THEIRSTACK', 'SRC_POSTINGS')

print(f"Total rows: {theirstack_profile['total_rows'].iloc[0]}")
print(f"Unique payload keys discovered: {len(theirstack_profile)}")
print()
display(theirstack_profile.drop(columns=['total_rows']))

Profiling TheirStack raw payload...
Total rows: 18
Unique payload keys discovered: 48



,PAYLOAD_KEY,NON_NULL_COUNT,population_pct,SAMPLE_VALUE
0,latitude,18,100.0,40.71427
1,id,18,100.0,698550590
2,keyword_slugs,18,100.0,"['job-descriptions', 'sensors-test-measurement', 'reporting-and-disclosure',..."
3,cities,18,100.0,[]
4,location,18,100.0,"New York, NY"
5,locations,18,100.0,"[{'address': None, 'admin1_code': 'NY', 'admin1_name': 'New York', 'admin2_c..."
6,long_location,18,100.0,"New York, NY"
7,longitude,18,100.0,-74.00597
8,manager_roles,18,100.0,[]
9,matching_phrases,18,100.0,[]


### TheirStack Investigation

In [17]:
# TheirStack: normalized_title — 100% present but blank sample.
#    Is it always blank or just sometimes?
# TheirStack normalized_title — confirm it's blank on ALL rows, not just the sample
df = run_query("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT_IF(RAW_PAYLOAD:normalized_title::STRING = '') AS blank_count,
        COUNT_IF(RAW_PAYLOAD:normalized_title::STRING != '') AS non_blank_count,
        COUNT_IF(RAW_PAYLOAD:normalized_title IS NULL) AS null_count
    FROM RAW.THEIRSTACK.SRC_POSTINGS
""")
print("TheirStack normalized_title population:")
display(df)

TheirStack normalized_title population:


,TOTAL_ROWS,BLANK_COUNT,NON_BLANK_COUNT,NULL_COUNT
0,18,18,0,0


In [8]:
# TheirStack: unpack company_object — what's actually in there?
#    Could have useful fields like company size, industry etc.
import json
df = run_query("SELECT RAW_PAYLOAD::STRING AS payload FROM RAW.THEIRSTACK.SRC_POSTINGS LIMIT 1")
sample = json.loads(df['PAYLOAD'].iloc[0])
print(json.dumps(sample.get('company_object', {}), indent=2))

{
  "alexa_ranking": null,
  "annual_revenue_usd": null,
  "annual_revenue_usd_readable": null,
  "apollo_id": null,
  "city": "New York",
  "company_keywords": [],
  "company_tags": [],
  "country": "United States",
  "country_code": "US",
  "domain": "hearst.com",
  "employee_count": 3935,
  "employee_count_range": "1001-5000",
  "founded_year": null,
  "funding_stage": null,
  "has_blurred_data": false,
  "id": "aoZzH+6F05/4DyWtEZ84U1g/7cYg0JeDC4wkzw6x+KVSq1PGjPZwpWjRlwdViuRv",
  "industry": "Broadcast Media Production and Distribution",
  "industry_id": 36,
  "investors": [],
  "is_recruiting_agency": false,
  "keyword_slugs": [],
  "last_funding_round_amount_readable": null,
  "last_funding_round_date": null,
  "linkedin_id": "476416",
  "linkedin_url": "https://www.linkedin.com/company/hearst-television/",
  "logo": "https://media.theirstack.com/company/logo/domain/hearst.com.jpeg",
  "long_description": "We\u2019re ambitious. We\u2019re smart. We want to be the best, to put our 

### Built In NYC Profiling

In [10]:
print('Profiling Built In raw payload...')
builtin_profile = profile_raw_source('RAW', 'BUILTIN', 'SRC_POSTINGS')

print(f"Total rows: {builtin_profile['total_rows'].iloc[0]}")
print(f"Unique payload keys discovered: {len(builtin_profile)}")
print()
display(builtin_profile.drop(columns=['total_rows']))

Profiling Built In raw payload...
Total rows: 18
Unique payload keys discovered: 19



,PAYLOAD_KEY,NON_NULL_COUNT,population_pct,SAMPLE_VALUE
0,@context,18,100.0,https://schema.org
1,employmentType,18,100.0,FULL_TIME
2,title,18,100.0,Investment Quant & Data Analyst
3,source_url,18,100.0,https://www.builtinnyc.com/job/investment-quant-data-analyst/9550551
4,scraped_at,18,100.0,2026-05-31T18:12:20.445986+00:00
5,industry,18,100.0,"['Fintech', 'Payments', 'Financial Services']"
6,identifier,18,100.0,"{'@type': 'PropertyValue', 'name': 'Resolution Life', 'value': '9550551'}"
7,@type,18,100.0,JobPosting
8,hiringOrganization,18,100.0,"{'@type': 'Organization', 'logo': {'@type': 'ImageObject', 'representativeOf..."
9,directApply,18,100.0,False


### Built In NYC Investigation

In [11]:
# Built In: unpack the nested objects — jobLocation, hiringOrganization, baseSalary, identifier
df = run_query("SELECT RAW_PAYLOAD::STRING AS payload FROM RAW.BUILTIN.SRC_POSTINGS LIMIT 3")
for _, row in df.iterrows():
    obj = json.loads(row['PAYLOAD'])
    print("--- jobLocation ---")
    print(json.dumps(obj.get('jobLocation', {}), indent=2))
    print("--- hiringOrganization ---")
    print(json.dumps(obj.get('hiringOrganization', {}), indent=2))
    print("--- baseSalary ---")
    print(json.dumps(obj.get('baseSalary', {}), indent=2))
    print("--- identifier ---")
    print(json.dumps(obj.get('identifier', {}), indent=2))
    print("="*60)

--- jobLocation ---
{
  "@type": "Place",
  "address": {
    "@type": "PostalAddress",
    "addressCountry": "USA",
    "addressLocality": "New York",
    "addressRegion": "New York"
  },
  "geo": {
    "@type": "GeoCoordinates",
    "latitude": 40.7130466,
    "longitude": -74.0072301
  }
}
--- hiringOrganization ---
{
  "@type": "Organization",
  "logo": {
    "@type": "ImageObject",
    "representativeOfPage": true,
    "url": "https://builtin.com/sites/www.builtin.com/files/2024-12/resolution_life_group_logo.JPEG"
  },
  "name": "Resolution Life",
  "sameAs": "https://builtin.com/company/resolution-life"
}
--- baseSalary ---
{
  "@type": "MonetaryAmount",
  "currency": "USD",
  "value": {
    "@type": "QuantitativeValue",
    "maxValue": 120000,
    "minValue": 90000,
    "unitText": "YEAR"
  }
}
--- identifier ---
{
  "@type": "PropertyValue",
  "name": "Resolution Life",
  "value": "9550551"
}
--- jobLocation ---
{
  "@type": "Place",
  "address": {
    "@type": "PostalAddress",


### Overall Investigations

In [13]:
# Cross-source deduplication signal — how much overlap is there?
#    Look for same job_title + company appearing in multiple sources
jsearch_jobs = run_query("""
    SELECT 
        RAW_PAYLOAD:job_title::STRING AS title,
        RAW_PAYLOAD:employer_name::STRING AS company
    FROM RAW.JSEARCH.SRC_POSTINGS
""")
theirstack_jobs = run_query("""
    SELECT
        RAW_PAYLOAD:job_title::STRING AS title,
        RAW_PAYLOAD:company::STRING AS company
    FROM RAW.THEIRSTACK.SRC_POSTINGS
""")
builtin_jobs = run_query("""
    SELECT
        RAW_PAYLOAD:title::STRING AS title,
        RAW_PAYLOAD:hiringOrganization:name::STRING AS company
    FROM RAW.BUILTIN.SRC_POSTINGS
""")

jsearch_jobs['source'] = 'jsearch'
theirstack_jobs['source'] = 'theirstack'
builtin_jobs['source'] = 'builtin'

all_jobs = pd.concat([jsearch_jobs, theirstack_jobs, builtin_jobs])
all_jobs.columns = ['title', 'company', 'source']
all_jobs['key'] = all_jobs['title'].str.strip().str.lower() + ' | ' + all_jobs['company'].str.strip().str.lower()

dupes = all_jobs[all_jobs.duplicated('key', keep=False)].sort_values('key')
print(f"Potentially duplicated jobs across sources: {dupes['key'].nunique()}")
display(dupes)

Potentially duplicated jobs across sources: 7


,title,company,source,key
74,Analytics & Data Engineering (Lead or Head),DualEntry,jsearch,analytics & data engineering (lead or head) | dualentry
101,Analytics & Data Engineering (Lead or Head),DualEntry,jsearch,analytics & data engineering (lead or head) | dualentry
96,"Analytics Engineer, Service Ops Analytics & AI",Capgemini,jsearch,"analytics engineer, service ops analytics & ai | capgemini"
15,"Analytics Engineer, Service Ops Analytics & AI",Capgemini,theirstack,"analytics engineer, service ops analytics & ai | capgemini"
54,Data Analyst - Economic Insights & Communications,Stripe,jsearch,data analyst - economic insights & communications | stripe
13,Data Analyst - Economic Insights & Communications,Stripe,builtin,data analyst - economic insights & communications | stripe
10,"Data Analyst, Operations Planning",Lyft,jsearch,"data analyst, operations planning | lyft"
13,"Data Analyst, Operations Planning",Lyft,jsearch,"data analyst, operations planning | lyft"
68,Data Scientist Analyst - Secondaries & Primaries,Ardian,jsearch,data scientist analyst - secondaries & primaries | ardian
16,Data Scientist Analyst - Secondaries & Primaries,Ardian,builtin,data scientist analyst - secondaries & primaries | ardian


In [21]:
# ID checks
# Confirm join keys match between raw payload and enrichment table for all 3 sources
checks = [
    ('JSearch',    'RAW.JSEARCH.SRC_POSTINGS',    'jsearch',    lambda p: p.get('job_id')),
    ('TheirStack', 'RAW.THEIRSTACK.SRC_POSTINGS', 'theirstack', lambda p: p.get('id')),
    ('BuiltIn',    'RAW.BUILTIN.SRC_POSTINGS',    'builtin',    lambda p: p.get('identifier', {}).get('value')),
]

for source_label, table, source_filter, id_extractor in checks:
    print(f"\n{'='*50}")
    print(f"{source_label}")

    # IDs from raw payload
    raw_df = run_query(f"SELECT RAW_PAYLOAD::STRING AS payload FROM {table}")
    raw_ids = set(str(id_extractor(json.loads(r))) for r in raw_df['PAYLOAD'])

    # IDs from enrichment table
    enrich_df = run_query(f"""
        SELECT JOB_ID FROM ENRICHED.PUBLIC.JOB_ENRICHMENT
        WHERE UPPER(SOURCE) LIKE '{source_filter.upper()}%'
    """)
    enrich_ids = set(enrich_df['JOB_ID'].astype(str))

    in_both      = raw_ids & enrich_ids
    raw_only     = raw_ids - enrich_ids
    enrich_only  = enrich_ids - raw_ids

    print(f"  Raw payload IDs:      {len(raw_ids)}")
    print(f"  Enrichment IDs:       {len(enrich_ids)}")
    print(f"  Matched (join works): {len(in_both)}")
    print(f"  Raw only (unenriched):{len(raw_only)}")
    print(f"  Enrichment only:      {len(enrich_only)}")


JSearch
  Raw payload IDs:      133
  Enrichment IDs:       133
  Matched (join works): 133
  Raw only (unenriched):0
  Enrichment only:      0

TheirStack
  Raw payload IDs:      18
  Enrichment IDs:       18
  Matched (join works): 18
  Raw only (unenriched):0
  Enrichment only:      0

BuiltIn
  Raw payload IDs:      18
  Enrichment IDs:       18
  Matched (join works): 18
  Raw only (unenriched):0
  Enrichment only:      0


In [ ]:
conn.close()
print('Connection closed.')